# PIPELINE 04: CUSTOMER 360 DATAMART (THE MERGER)

**Mục tiêu của Notebook:**
Ghép nối 2 nguồn dữ liệu đã được làm giàu (Enriched) từ hành vi **Tìm kiếm (Search)** và **Xem phim (Content)**.

**Chiến lược Gộp (Join Strategy):**
Sử dụng **Inner Join** dựa trên `user_id`. 
Lý do: Tập trung phân tích sâu vào tệp khách hàng chất lượng cao - những người vừa có ý định tìm kiếm rõ ràng, vừa có hành vi tiêu thụ nội dung thực tế trên nền tảng.

In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os
from pyspark.sql import Window


# Cấu hình đường dẫn nội bộ
BASE_DIR = "/Users/hoaithuong/Desktop/Customer-Behavior-Analysis-Pipeline"
SEARCH_PATH = f"{BASE_DIR}/data/processed/Master_Table_Log_Search.parquet"
CONTENT_PATH = f"{BASE_DIR}/data/processed/Master_Content_Enriched.parquet"

# Cấu hình MySQL Driver (Kiểm tra lại đường dẫn jar trên máy bạn)
JDBC_DRIVER_PATH = "/opt/homebrew/Cellar/apache-spark/4.0.1_1/libexec/jars/mysql-connector-java-8.0.28.jar"

# Khởi tạo Spark với JDBC Driver
spark = SparkSession.builder \
    .appName("Test_Final_Merger") \
    .config("spark.driver.memory", "4g") \
    .config("spark.jars", JDBC_DRIVER_PATH) \
    .getOrCreate()

print(">>> Spark Session đã sẵn sàng để Join dữ liệu!")

26/04/08 17:50:53 WARN Utils: Your hostname, Ryms.local resolves to a loopback address: 127.0.0.1; using 192.168.1.234 instead (on interface en0)
26/04/08 17:50:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/04/08 17:50:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/08 17:50:54 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


>>> Spark Session đã sẵn sàng để Join dữ liệu!


### Bước 1: Đọc 2 bảng Master Data
Nạp 2 bảng dữ liệu từ lớp `processed/` lên RAM.

In [11]:
print(">>> Đang nạp Master_Search và Master_Content...")
df_search = spark.read.parquet(SEARCH_PATH)
df_content = spark.read.parquet(CONTENT_PATH)

print(f"- Số lượng User bên Search: {df_search.count()}")
print(f"- Số lượng User bên Content: {df_content.count()}")

>>> Đang nạp Master_Search và Master_Content...
- Số lượng User bên Search: 5000
- Số lượng User bên Content: 5000


In [12]:
df_content.show(5)

+---------+-----------------+-----------------+--------------+---------------+--------------+-----------+--------------------+------+
|  user_id|Total_Truyền_Hình|Total_Phim_Truyện|Total_Giải_Trí|Total_Thiếu_Nhi|Total_Thể_Thao|  MostWatch|               Taste|Active|
+---------+-----------------+-----------------+--------------+---------------+--------------+-----------+--------------------+------+
|SGH860457|            22560|                0|             0|              0|             0|Truyền Hình|         Truyền Hình| 13.33|
|HNH629200|           992975|                0|             0|            407|             0|Truyền Hình|Truyền Hình - Thi...| 100.0|
|HNH429678|           141508|           360239|             0|              0|             0|Phim Truyện|Truyền Hình - Phi...| 100.0|
|SGH525273|           118201|             1617|             0|              0|             0|Truyền Hình|Truyền Hình - Phi...| 46.67|
|HNAAC7001|           129083|             2156|             0|

In [13]:
df_search.show(5)

+--------+--------------------+--------------+--------------------+--------------+-------------+--------------+--------------------+
| user_id|      most_search_t6|   category_t6|      most_search_t7|   category_t7|  user_status|trending_types|            Previous|
+--------+--------------------+--------------+--------------------+--------------+-------------+--------------+--------------------+
|49081622|kung fu masters o...|  Cartoon_Kids|      johnny english|     Education|Retained_User|       changed|Cartoon_Kids -> E...|
| 5839440|   xem phim việt nam|Movies_General|tập dưỡng sinh tr...|       K-Drama|Retained_User|       changed|Movies_General ->...|
|45774316|thiên kim bảo yến...|       C-Drama|one way ticket | ...|Movies_General|Retained_User|       changed|C-Drama -> Movies...|
| 5979363|         một con vịt|         Music|         một con vịt|         Music|Retained_User|     unchanged|               Music|
|42181171|bống bống bang ba...|  Cartoon_Kids|  oppa gangnam style|  

### Bước 2: Thực hiện Inner Join & Xử lý giá trị rỗng
Vì dùng Inner Join, chúng ta đã loại bỏ được các User lệch pha. Tuy nhiên, vẫn cần hàm `fillna` như một chốt chặn an toàn (Safety Net) để xử lý các giá trị `NULL` có thể đã tồn tại từ các bước trước trong từng bảng riêng lẻ.

In [14]:
# 1. Tạo một "khung" để đếm số thứ tự dòng
windowSpec = Window.orderBy(F.monotonically_increasing_id())

# 2. Đánh số thứ tự dòng cho từng bảng
df_search_with_idx = df_search.withColumn("row_idx", F.row_number().over(windowSpec))
df_content_with_idx = df_content.drop("user_id").withColumn("row_idx", F.row_number().over(windowSpec))

# 3. Join 2 bảng dựa trên cột 'row_idx' vừa tạo, sau đó xóa luôn cột này đi cho gọn
df_360 = df_search_with_idx.join(df_content_with_idx, on="row_idx", how="inner").drop("row_idx")

df_360.show(10)

+--------+--------------------+--------------+--------------------+--------------+-------------+--------------+--------------------+-----------------+-----------------+--------------+---------------+--------------+-----------+--------------------+------+
| user_id|      most_search_t6|   category_t6|      most_search_t7|   category_t7|  user_status|trending_types|            Previous|Total_Truyền_Hình|Total_Phim_Truyện|Total_Giải_Trí|Total_Thiếu_Nhi|Total_Thể_Thao|  MostWatch|               Taste|Active|
+--------+--------------------+--------------+--------------------+--------------+-------------+--------------+--------------------+-----------------+-----------------+--------------+---------------+--------------+-----------+--------------------+------+
|49081622|kung fu masters o...|  Cartoon_Kids|      johnny english|     Education|Retained_User|       changed|Cartoon_Kids -> E...|            22560|                0|             0|              0|             0|Truyền Hình|         

26/04/08 17:56:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/08 17:56:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/08 17:56:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/08 17:56:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/08 17:56:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/08 17:56:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/08 1

In [15]:
df_360.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- most_search_t6: string (nullable = true)
 |-- category_t6: string (nullable = true)
 |-- most_search_t7: string (nullable = true)
 |-- category_t7: string (nullable = true)
 |-- user_status: string (nullable = true)
 |-- trending_types: string (nullable = true)
 |-- Previous: string (nullable = true)
 |-- Total_Truyền_Hình: long (nullable = true)
 |-- Total_Phim_Truyện: long (nullable = true)
 |-- Total_Giải_Trí: long (nullable = true)
 |-- Total_Thiếu_Nhi: long (nullable = true)
 |-- Total_Thể_Thao: long (nullable = true)
 |-- MostWatch: string (nullable = true)
 |-- Taste: string (nullable = true)
 |-- Active: double (nullable = true)



In [16]:
df_360.cache()

print(f">>> TỔNG SỐ KHÁCH HÀNG 360 ĐỘ (INNER JOIN): {df_360.count()}")
print(">>> Xem thử 10 dòng tiêu biểu:")
df_360.show(10, truncate=False)

26/04/08 17:56:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/08 17:56:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/08 17:56:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/08 17:56:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/08 17:56:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/08 17:56:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/08 1

>>> TỔNG SỐ KHÁCH HÀNG 360 ĐỘ (INNER JOIN): 5000
>>> Xem thử 10 dòng tiêu biểu:
+--------+-------------------------------------------------+--------------+------------------------------------------------------+--------------+-------------+--------------+-------------------------+-----------------+-----------------+--------------+---------------+--------------+-----------+-------------------------------------+------+
|user_id |most_search_t6                                   |category_t6   |most_search_t7                                        |category_t7   |user_status  |trending_types|Previous                 |Total_Truyền_Hình|Total_Phim_Truyện|Total_Giải_Trí|Total_Thiếu_Nhi|Total_Thể_Thao|MostWatch  |Taste                                |Active|
+--------+-------------------------------------------------+--------------+------------------------------------------------------+--------------+-------------+--------------+-------------------------+-----------------+-----------------+----